# Project XIA: CS-SHAP feature selection for multiclass IoT/IIoT intrusion detection

**Author:** Abdul Hafiz Abdullai (Independent Researcher, Ghana)  
**ORCID:** https://orcid.org/0009-0001-6995-0747

This is the complete analysis used for the Project XIA manuscript: data audit and leakage control, a development/locked-test split, XGBoost baseline, class-sensitive SHAP ranking, stability analysis, nested cross-validation, frozen feature sets, and one final locked-test evaluation.

## Reproduction notes

- Designed for Kaggle with Edge-IIoTset attached at the path used below.
- Run from top to bottom in a fresh session; outputs go to `/kaggle/working/`.
- The external dataset is not redistributed here.
- Never use the locked test set for tuning, feature selection, or model choice.


In [ ]:
import os

folder = (
    "/kaggle/input/edgeiiotset-cyber-security-dataset-of-iot-iiot/"
    "Edge-IIoTset dataset/Selected dataset for ML and DL"
)

for filename in os.listdir(folder):
    path = os.path.join(folder, filename)
    size_gb = os.path.getsize(path) / (1024**3)
    print(f"{filename}: {size_gb:.2f} GB")


In [ ]:
import pandas as pd
folder = (
    "/kaggle/input/edgeiiotset-cyber-security-dataset-of-iot-iiot/"
    "Edge-IIoTset dataset/Selected dataset for ML and DL"
)
df = pd.read_csv(
    f"{folder}/ML-EdgeIIoT-dataset.csv",
    low_memory=False
)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nAttack types:")
print(df["Attack_type"].value_counts(dropna=False))

In [ ]:
import pandas as pd
import numpy as np

print("Dataset shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())
print("Total missing values:", df.isna().sum().sum())

numeric_df = df.select_dtypes(include=np.number)

print("Numeric columns:", numeric_df.shape[1])
print("Categorical/non-numeric columns:", df.shape[1] - numeric_df.shape[1])

print(
    "Positive infinity values:",
    np.isposinf(numeric_df).sum().sum()
)

print(
    "Negative infinity values:",
    np.isneginf(numeric_df).sum().sum()
)

audit = pd.DataFrame({
    "dtype": df.dtypes,
    "missing": df.isna().sum(),
    "missing_percentage": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(dropna=False)
})

display(audit)

class_distribution = pd.DataFrame({
    "count": df["Attack_type"].value_counts(),
    "percentage": (
        df["Attack_type"].value_counts(normalize=True) * 100
    ).round(2)
})

display(class_distribution)

In [ ]:
label_check = pd.crosstab(
    df["Attack_type"],
    df["Attack_label"]
)

display(label_check)

inconsistent_labels = df[
    ((df["Attack_type"] == "Normal") & (df["Attack_label"] != 0)) |
    ((df["Attack_type"] != "Normal") & (df["Attack_label"] != 1))
]

print("Inconsistent labels:", len(inconsistent_labels))

In [ ]:
import pandas as pd

target_cols = ["Attack_label", "Attack_type"]
feature_cols = [c for c in df.columns if c not in target_cols]

# 1. Exact duplicate rows, including labels
exact_duplicate_mask = df.duplicated(keep=False)
exact_duplicates = df.loc[exact_duplicate_mask]

print("Rows belonging to exact duplicate groups:",
      exact_duplicate_mask.sum())

print("Extra exact duplicate copies:",
      df.duplicated().sum())

print("\nExact duplicate rows by attack class:")
display(
    exact_duplicates["Attack_type"]
    .value_counts()
    .to_frame("rows_in_duplicate_groups")
)

# 2. Duplicate predictor vectors, ignoring both targets
feature_duplicate_mask = df.duplicated(
    subset=feature_cols,
    keep=False
)

print("Rows with duplicated predictor vectors:",
      feature_duplicate_mask.sum())

print("Extra copies based on predictors:",
      df.duplicated(subset=feature_cols).sum())

# 3. Check whether identical predictor vectors have conflicting labels
conflict_check = (
    df.groupby(feature_cols, dropna=False)["Attack_type"]
      .nunique()
)

conflicting_feature_groups = conflict_check[
    conflict_check > 1
]

print("Predictor-identical groups with conflicting Attack_type:",
      len(conflicting_feature_groups))

In [ ]:
predictors = df.drop(columns=["Attack_label", "Attack_type"])

feature_summary = pd.DataFrame({
    "dtype": predictors.dtypes.astype(str),
    "unique_values": predictors.nunique(dropna=False),
    "most_common_count": predictors.apply(
        lambda x: x.value_counts(dropna=False).iloc[0]
    )
})

feature_summary["most_common_percentage"] = (
    feature_summary["most_common_count"] / len(predictors) * 100
).round(2)

constant_features = feature_summary[
    feature_summary["unique_values"] <= 1
]

near_constant_features = feature_summary[
    (feature_summary["unique_values"] > 1) &
    (feature_summary["most_common_percentage"] >= 99.0)
].sort_values("most_common_percentage", ascending=False)

print("Constant features:")
display(constant_features)

print("Near-constant features (at least 99% one value):")
display(near_constant_features)

print("Categorical features:")
display(
    feature_summary[
        feature_summary["dtype"] == "object"
    ].sort_values("unique_values", ascending=False)
)

In [ ]:
suspect_features = [
    "frame.time",
    "ip.src_host",
    "ip.dst_host"
]

for column in suspect_features:
    print(f"\n===== {column} =====")

    summary = (
        df.groupby("Attack_type")[column]
          .agg(["count", "nunique"])
          .sort_values("nunique", ascending=False)
    )

    display(summary)

    # Number of values exclusive to only one attack class
    classes_per_value = (
        df.groupby(column)["Attack_type"]
          .nunique()
    )

    exclusive_values = (classes_per_value == 1).sum()

    print("Total unique values:", df[column].nunique())
    print("Values appearing in only one class:", exclusive_values)
    print(
        "Exclusive-value percentage:",
        round(exclusive_values / df[column].nunique() * 100, 2),
        "%"
    )

In [ ]:
# Preserve the original dataset
df_raw = df.copy()

constant_features = [
    "icmp.unused",
    "http.tls_port",
    "dns.qry.type",
    "dns.retransmit_request_in",
    "mqtt.msg_decoded_as",
    "mbtcp.len",
    "mbtcp.trans_id",
    "mbtcp.unit_id"
]

identifier_features = [
    "frame.time",
    "ip.src_host",
    "ip.dst_host"
]

# Remove exact duplicate copies while retaining one occurrence
df_controlled = df_raw.drop_duplicates().copy()

# Separate multiclass target
y = df_controlled["Attack_type"].copy()

# Attack_label must not enter the predictor matrix
X = df_controlled.drop(
    columns=[
        "Attack_type",
        "Attack_label",
        *identifier_features,
        *constant_features
    ]
).copy()

print("Original dataset:", df_raw.shape)
print("After deduplication:", df_controlled.shape)
print("Predictor matrix:", X.shape)
print("Target:", y.shape)

print("\nClass distribution after deduplication:")
display(
    pd.DataFrame({
        "count": y.value_counts(),
        "percentage": (
            y.value_counts(normalize=True) * 100
        ).round(3)
    })
)

print("\nRemaining predictor types:")
print(X.dtypes.value_counts())

In [ ]:
object_columns = X.select_dtypes(include="object").columns.tolist()

results = []

for column in object_columns:
    values_per_class = (
        df_controlled.groupby(column, dropna=False)["Attack_type"]
        .nunique()
    )

    exclusive_values = int((values_per_class == 1).sum())
    total_unique = int(df_controlled[column].nunique(dropna=False))

    numeric_conversion = pd.to_numeric(
        df_controlled[column],
        errors="coerce"
    )

    numeric_success = numeric_conversion.notna().mean() * 100

    results.append({
        "feature": column,
        "unique_values": total_unique,
        "class_exclusive_values": exclusive_values,
        "exclusive_percentage": round(
            exclusive_values / total_unique * 100, 2
        ),
        "numeric_conversion_success": round(numeric_success, 2)
    })

object_audit = pd.DataFrame(results).sort_values(
    ["exclusive_percentage", "unique_values"],
    ascending=False
)

display(object_audit)

In [ ]:
for column in object_columns:
    print(f"\n===== {column} =====")
    print(df_controlled[column].value_counts(dropna=False).head(10))

In [ ]:
import numpy as np
import pandas as pd

content_and_address_features = [
    "tcp.options",
    "tcp.payload",
    "http.request.full_uri",
    "http.request.uri.query",
    "mqtt.msg",
    "http.file_data",
    "arp.dst.proto_ipv4",
    "arp.src.proto_ipv4"
]

X_primary = X.drop(
    columns=content_and_address_features
).copy()

# Convert columns that should be numeric
numeric_conversion_columns = [
    "tcp.srcport",
    "dns.qry.name.len"
]

conversion_report = []

for column in numeric_conversion_columns:
    before_missing = X_primary[column].isna().sum()

    converted = pd.to_numeric(
        X_primary[column],
        errors="coerce"
    )

    after_missing = converted.isna().sum()

    conversion_report.append({
        "feature": column,
        "new_missing_values": after_missing - before_missing,
        "total_missing_after_conversion": after_missing
    })

    X_primary[column] = converted

print("Primary predictor matrix:", X_primary.shape)
print("\nConversion report:")
display(pd.DataFrame(conversion_report))

print("\nPredictor types:")
print(X_primary.dtypes.value_counts())

print("\nRemaining categorical columns:")
print(X_primary.select_dtypes(include="object").columns.tolist())

In [ ]:
categorical_features = (
    X_primary.select_dtypes(include="object")
    .columns.tolist()
)

numeric_features = (
    X_primary.select_dtypes(exclude="object")
    .columns.tolist()
)

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Total predictors:",
      len(numeric_features) + len(categorical_features))

print("\nCategorical feature cardinalities:")
display(
    X_primary[categorical_features]
    .nunique(dropna=False)
    .sort_values()
    .to_frame("unique_values")
)

print("\nMissing values created by numeric conversion:")
display(
    X_primary[numeric_conversion_columns]
    .isna()
    .sum()
    .to_frame("missing_values")
)

In [ ]:
for column in ["tcp.srcport", "dns.qry.name.len"]:
    converted = pd.to_numeric(X[column], errors="coerce")
    invalid_mask = converted.isna() & X[column].notna()

    print(f"\n===== {column} =====")
    print("Invalid entries:", invalid_mask.sum())
    display(
        X.loc[invalid_mask, column]
        .value_counts(dropna=False)
        .head(20)
        .to_frame("count")
    )

## Development and locked-test split

The test partition is isolated here until final evaluation.

In [ ]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TEST_SIZE = 0.20

X_development, X_test, y_development, y_test = train_test_split(
    X_primary,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Development features:", X_development.shape)
print("Locked test features:", X_test.shape)
print("Development target:", y_development.shape)
print("Locked test target:", y_test.shape)

In [ ]:
split_distribution = pd.concat(
    [
        y.value_counts().rename("complete_dataset"),
        y_development.value_counts().rename("development"),
        y_test.value_counts().rename("locked_test")
    ],
    axis=1
)

split_distribution["test_percentage"] = (
    split_distribution["locked_test"] /
    split_distribution["complete_dataset"] * 100
).round(2)

display(split_distribution)

In [ ]:
index_overlap = set(X_development.index).intersection(
    set(X_test.index)
)

print("Index overlap:", len(index_overlap))
print(
    "All classes in development:",
    y_development.nunique()
)

print(
    "All classes in locked test:",
    y_test.nunique()
)

In [ ]:
split_register = pd.DataFrame({
    "original_index": list(X_development.index) + list(X_test.index),
    "split": (
        ["development"] * len(X_development) +
        ["locked_test"] * len(X_test)
    )
})

split_register.to_csv(
    "/kaggle/working/project_xia_split_register.csv",
    index=False
)

print("Split register saved.")
display(split_register["split"].value_counts())

## Preprocessing and encoding

Transformations are fitted inside pipelines to prevent leakage.

In [ ]:
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

print("Scikit-learn version:", sklearn.__version__)

categorical_features = (
    X_primary.select_dtypes(include="object")
    .columns.tolist()
)

numeric_features = (
    X_primary.select_dtypes(exclude="object")
    .columns.tolist()
)

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse=True
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ],
    remainder="drop"
)

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Preprocessor created successfully but has not been fitted.")

In [ ]:
import inspect
from sklearn.preprocessing import OneHotEncoder

encoder_parameters = inspect.signature(
    OneHotEncoder
).parameters

if "sparse_output" in encoder_parameters:
    one_hot_encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=True
    )
else:
    one_hot_encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse=True
    )

print("Compatible OneHotEncoder created.")

In [ ]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            one_hot_encoder
        )
    ]
)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

X_train, X_validation, y_train_text, y_validation_text = train_test_split(
    X_development,
    y_development,
    test_size=0.20,
    random_state=42,
    stratify=y_development
)

label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(y_train_text)
y_validation = label_encoder.transform(y_validation_text)

print("Training set:", X_train.shape)
print("Validation set:", X_validation.shape)
print("Classes:", list(label_encoder.classes_))

## XGBoost baseline and cross-validation

In [ ]:
import xgboost as xgb

print("XGBoost version:", xgb.__version__)

In [ ]:
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

baseline_model = XGBClassifier(
    objective="multi:softprob",
    num_class=len(label_encoder.classes_),
    n_estimators=250,
    max_depth=6,
    learning_rate=0.10,
    subsample=0.80,
    colsample_bytree=0.80,
    min_child_weight=1,
    reg_alpha=0.0,
    reg_lambda=1.0,
    tree_method="hist",
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

baseline_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", baseline_model)
    ]
)

print("Baseline pipeline created.")

In [ ]:
import time

start_time = time.perf_counter()

baseline_pipeline.fit(
    X_train,
    y_train
)

training_time = time.perf_counter() - start_time

print(
    f"Training completed in {training_time:.2f} seconds."
)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
    matthews_corrcoef
)

start_time = time.perf_counter()

validation_predictions = baseline_pipeline.predict(
    X_validation
)

inference_time = time.perf_counter() - start_time

accuracy = accuracy_score(
    y_validation,
    validation_predictions
)

macro_f1 = f1_score(
    y_validation,
    validation_predictions,
    average="macro"
)

weighted_f1 = f1_score(
    y_validation,
    validation_predictions,
    average="weighted"
)

balanced_accuracy = balanced_accuracy_score(
    y_validation,
    validation_predictions
)

mcc = matthews_corrcoef(
    y_validation,
    validation_predictions
)

print("INITIAL XGBOOST BASELINE")
print("--------------------------")
print(f"Accuracy:             {accuracy:.4f}")
print(f"Macro F1:             {macro_f1:.4f}")
print(f"Weighted F1:          {weighted_f1:.4f}")
print(f"Balanced accuracy:    {balanced_accuracy:.4f}")
print(f"Multiclass MCC:       {mcc:.4f}")
print(f"Training time:        {training_time:.2f} seconds")
print(f"Inference time:       {inference_time:.2f} seconds")
print(
    "Milliseconds/record:",
    round(
        inference_time / len(X_validation) * 1000,
        6
    )
)

In [ ]:
print(
    classification_report(
        y_validation,
        validation_predictions,
        target_names=label_encoder.classes_,
        digits=4,
        zero_division=0
    )
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_validation,
    validation_predictions
)

plt.figure(figsize=(14, 11))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)

plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.title(
    "PROJECT XIA: Initial XGBoost Validation Confusion Matrix"
)
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
import time
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

# Encode the entire development target using the existing mapping
y_development_encoded = label_encoder.transform(
    y_development
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fold_results = []
oof_predictions = np.full(
    len(y_development_encoded),
    -1,
    dtype=int
)

for fold_number, (train_indices, validation_indices) in enumerate(
    cv.split(X_development, y_development_encoded),
    start=1
):
    print(f"Running fold {fold_number}/5...")

    X_fold_train = X_development.iloc[train_indices]
    X_fold_validation = X_development.iloc[validation_indices]

    y_fold_train = y_development_encoded[train_indices]
    y_fold_validation = y_development_encoded[
        validation_indices
    ]

    fold_model = XGBClassifier(
        objective="multi:softprob",
        num_class=len(label_encoder.classes_),
        n_estimators=250,
        max_depth=6,
        learning_rate=0.10,
        subsample=0.80,
        colsample_bytree=0.80,
        min_child_weight=1,
        reg_alpha=0.0,
        reg_lambda=1.0,
        tree_method="hist",
        eval_metric="mlogloss",
        use_label_encoder=False,
        random_state=42,
        n_jobs=-1
    )

    fold_pipeline = Pipeline(
        steps=[
            ("preprocessor", clone(preprocessor)),
            ("model", fold_model)
        ]
    )

    start_time = time.perf_counter()

    fold_pipeline.fit(
        X_fold_train,
        y_fold_train
    )

    fold_training_time = time.perf_counter() - start_time

    fold_predictions = fold_pipeline.predict(
        X_fold_validation
    )

    oof_predictions[validation_indices] = fold_predictions

    fold_results.append({
        "fold": fold_number,
        "accuracy": accuracy_score(
            y_fold_validation,
            fold_predictions
        ),
        "macro_f1": f1_score(
            y_fold_validation,
            fold_predictions,
            average="macro"
        ),
        "weighted_f1": f1_score(
            y_fold_validation,
            fold_predictions,
            average="weighted"
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_fold_validation,
            fold_predictions
        ),
        "mcc": matthews_corrcoef(
            y_fold_validation,
            fold_predictions
        ),
        "training_seconds": fold_training_time
    })

cv_results = pd.DataFrame(fold_results)

display(cv_results.round(4))

In [ ]:
metric_columns = [
    "accuracy",
    "macro_f1",
    "weighted_f1",
    "balanced_accuracy",
    "mcc",
    "training_seconds"
]

cv_summary = pd.DataFrame({
    "mean": cv_results[metric_columns].mean(),
    "standard_deviation": cv_results[metric_columns].std(),
    "minimum": cv_results[metric_columns].min(),
    "maximum": cv_results[metric_columns].max()
})

print("FIVE-FOLD BASELINE SUMMARY")
display(cv_summary.round(4))

In [ ]:
from sklearn.metrics import classification_report

print("OUT-OF-FOLD CLASSIFICATION REPORT\n")

print(
    classification_report(
        y_development_encoded,
        oof_predictions,
        target_names=label_encoder.classes_,
        digits=4,
        zero_division=0
    )
)

In [ ]:
cv_results.to_csv(
    "/kaggle/working/project_xia_baseline_cv_folds.csv",
    index=False
)

cv_summary.to_csv(
    "/kaggle/working/project_xia_baseline_cv_summary.csv"
)

oof_results = pd.DataFrame({
    "original_index": X_development.index,
    "actual_class": label_encoder.inverse_transform(
        y_development_encoded
    ),
    "predicted_class": label_encoder.inverse_transform(
        oof_predictions
    )
})

oof_results.to_csv(
    "/kaggle/working/project_xia_baseline_oof_predictions.csv",
    index=False
)

print("Cross-validation evidence saved.")

## SHAP analysis

Class-specific TreeSHAP contributions are aggregated to original features.

In [ ]:
import time

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

y_development_encoded = label_encoder.transform(
    y_development
)

full_baseline_model = XGBClassifier(
    objective="multi:softprob",
    num_class=len(label_encoder.classes_),
    n_estimators=250,
    max_depth=6,
    learning_rate=0.10,
    subsample=0.80,
    colsample_bytree=0.80,
    min_child_weight=1,
    reg_alpha=0.0,
    reg_lambda=1.0,
    tree_method="hist",
    eval_metric="mlogloss",
    use_label_encoder=False,
    random_state=42,
    n_jobs=-1
)

full_baseline_pipeline = Pipeline(
    steps=[
        ("preprocessor", clone(preprocessor)),
        ("model", full_baseline_model)
    ]
)

start_time = time.perf_counter()

full_baseline_pipeline.fit(
    X_development,
    y_development_encoded
)

full_training_time = time.perf_counter() - start_time

print(
    f"Full development model trained in "
    f"{full_training_time:.2f} seconds."
)

In [ ]:
import pandas as pd

shap_sample_indices = []

for class_number, class_name in enumerate(
    label_encoder.classes_
):
    class_indices = y_development[
        y_development == class_name
    ].index

    sample_size = min(
        400,
        len(class_indices)
    )

    sampled_indices = (
        pd.Series(class_indices)
        .sample(
            n=sample_size,
            random_state=42 + class_number
        )
        .tolist()
    )

    shap_sample_indices.extend(sampled_indices)

X_shap = X_development.loc[
    shap_sample_indices
].copy()

y_shap = y_development.loc[
    shap_sample_indices
].copy()

print("SHAP sample shape:", X_shap.shape)

display(
    y_shap.value_counts()
    .reindex(label_encoder.classes_)
    .to_frame("SHAP records")
)

In [ ]:
fitted_preprocessor = (
    full_baseline_pipeline
    .named_steps["preprocessor"]
)

fitted_xgboost = (
    full_baseline_pipeline
    .named_steps["model"]
)

X_shap_transformed = fitted_preprocessor.transform(
    X_shap
)

try:
    transformed_feature_names = (
        fitted_preprocessor
        .get_feature_names_out()
    )
except AttributeError:
    transformed_feature_names = [
        f"transformed_feature_{i}"
        for i in range(X_shap_transformed.shape[1])
    ]

print(
    "Transformed SHAP matrix:",
    X_shap_transformed.shape
)

print(
    "Transformed feature names:",
    len(transformed_feature_names)
)

In [ ]:
import joblib

joblib.dump(
    full_baseline_pipeline,
    "/kaggle/working/project_xia_full_baseline_pipeline.joblib"
)

joblib.dump(
    label_encoder,
    "/kaggle/working/project_xia_label_encoder.joblib"
)

print("Baseline model and label encoder saved.")

In [ ]:
print("SHAP sample:", X_shap.shape)
print("Transformed SHAP sample:", X_shap_transformed.shape)
print("Classes:", len(label_encoder.classes_))

In [ ]:
import numpy as np
import pandas as pd

fitted_preprocessor = (
    full_baseline_pipeline
    .named_steps["preprocessor"]
)

fitted_xgboost = (
    full_baseline_pipeline
    .named_steps["model"]
)

fitted_encoder = (
    fitted_preprocessor
    .named_transformers_["categorical"]
    .named_steps["encoder"]
)

# ColumnTransformer outputs numeric columns first
original_feature_map = list(numeric_features)

# Add one parent feature for every one-hot encoded column
for feature, categories in zip(
    categorical_features,
    fitted_encoder.categories_
):
    original_feature_map.extend(
        [feature] * len(categories)
    )

print(
    "Transformed columns:",
    X_shap_transformed.shape[1]
)

print(
    "Mapped columns:",
    len(original_feature_map)
)

assert (
    len(original_feature_map) ==
    X_shap_transformed.shape[1]
), "Feature mapping does not match the transformed matrix."

print("Feature mapping verified.")

In [ ]:
import xgboost as xgb
import time

booster = fitted_xgboost.get_booster()

shap_dmatrix = xgb.DMatrix(
    X_shap_transformed
)

start_time = time.perf_counter()

shap_contributions = booster.predict(
    shap_dmatrix,
    pred_contribs=True
)

shap_time = time.perf_counter() - start_time

print("Raw contribution shape:", shap_contributions.shape)
print(f"TreeSHAP time: {shap_time:.2f} seconds")

In [ ]:
if shap_contributions.ndim != 3:
    raise ValueError(
        "Unexpected TreeSHAP shape: "
        f"{shap_contributions.shape}"
    )

feature_contributions = shap_contributions[:, :, :-1]

print(
    "Feature contribution matrix:",
    feature_contributions.shape
)

assert (
    feature_contributions.shape[2] ==
    len(original_feature_map)
)

In [ ]:
y_shap_encoded = label_encoder.transform(y_shap)

# Conventional global multiclass importance
transformed_global_importance = (
    np.abs(feature_contributions)
    .mean(axis=(0, 1))
)

# Class-specific importance for the correct class output
class_specific_importance = []

for class_number, class_name in enumerate(
    label_encoder.classes_
):
    class_mask = y_shap_encoded == class_number

    importance = np.abs(
        feature_contributions[
            class_mask,
            class_number,
            :
        ]
    ).mean(axis=0)

    class_specific_importance.append(importance)

class_specific_importance = np.vstack(
    class_specific_importance
)

# Every class receives equal weight
transformed_balanced_importance = (
    class_specific_importance.mean(axis=0)
)

minority_classes = [
    "Fingerprinting",
    "MITM"
]

minority_class_numbers = [
    list(label_encoder.classes_).index(class_name)
    for class_name in minority_classes
]

transformed_minority_importance = (
    class_specific_importance[
        minority_class_numbers
    ].mean(axis=0)
)

print(
    "Class-specific matrix:",
    class_specific_importance.shape
)

In [ ]:
def aggregate_to_original_features(
    transformed_scores,
    original_map
):
    score_table = pd.DataFrame({
        "feature": original_map,
        "importance": transformed_scores
    })

    return (
        score_table.groupby(
            "feature",
            as_index=False
        )["importance"]
        .sum()
        .set_index("feature")["importance"]
    )

global_importance = aggregate_to_original_features(
    transformed_global_importance,
    original_feature_map
)

balanced_importance = aggregate_to_original_features(
    transformed_balanced_importance,
    original_feature_map
)

minority_importance = aggregate_to_original_features(
    transformed_minority_importance,
    original_feature_map
)

shap_rankings = pd.concat(
    [
        global_importance.rename("global_shap"),
        balanced_importance.rename(
            "class_balanced_shap"
        ),
        minority_importance.rename(
            "minority_diagnostic_shap"
        )
    ],
    axis=1
)

for column in [
    "global_shap",
    "class_balanced_shap",
    "minority_diagnostic_shap"
]:
    shap_rankings[
        column.replace("shap", "rank")
    ] = (
        shap_rankings[column]
        .rank(
            ascending=False,
            method="min"
        )
        .astype(int)
    )

shap_rankings = shap_rankings.sort_values(
    "global_shap",
    ascending=False
)

display(shap_rankings.head(20))

In [ ]:
shap_rankings.to_csv(
    "/kaggle/working/"
    "project_xia_initial_shap_rankings.csv"
)

class_specific_original = {}

for class_number, class_name in enumerate(
    label_encoder.classes_
):
    class_specific_original[class_name] = (
        aggregate_to_original_features(
            class_specific_importance[class_number],
            original_feature_map
        )
    )

class_specific_table = pd.DataFrame(
    class_specific_original
)

class_specific_table.to_csv(
    "/kaggle/working/"
    "project_xia_class_specific_shap.csv"
)

print("SHAP evidence files saved.")

In [ ]:
import matplotlib.pyplot as plt

score_columns = [
    (
        "global_shap",
        "Conventional Global SHAP"
    ),
    (
        "class_balanced_shap",
        "Equal-Class Balanced SHAP"
    ),
    (
        "minority_diagnostic_shap",
        "MITM and Fingerprinting SHAP"
    )
]

fig, axes = plt.subplots(
    1,
    3,
    figsize=(20, 8)
)

for axis, (column, title) in zip(
    axes,
    score_columns
):
    top_features = (
        shap_rankings[column]
        .sort_values()
        .tail(15)
    )

    axis.barh(
        top_features.index,
        top_features.values
    )

    axis.set_title(title)
    axis.set_xlabel("Mean absolute TreeSHAP contribution")

plt.suptitle(
    "PROJECT XIA: Initial TreeSHAP Feature Rankings",
    fontsize=16
)

plt.tight_layout()

plt.savefig(
    "/kaggle/working/"
    "project_xia_initial_shap_rankings.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
NATURAL_SAMPLE_SIZE = 6000

X_natural_shap = X_development.sample(
    n=min(
        NATURAL_SAMPLE_SIZE,
        len(X_development)
    ),
    random_state=42
)

y_natural_shap = y_development.loc[
    X_natural_shap.index
]

print(
    "Natural SHAP sample:",
    X_natural_shap.shape
)

natural_distribution = pd.DataFrame({
    "sample_count": (
        y_natural_shap.value_counts()
    ),
    "sample_percentage": (
        y_natural_shap
        .value_counts(normalize=True)
        .mul(100)
        .round(3)
    )
})

display(
    natural_distribution.reindex(
        label_encoder.classes_
    )
)

In [ ]:
X_natural_transformed = (
    fitted_preprocessor.transform(
        X_natural_shap
    )
)

natural_dmatrix = xgb.DMatrix(
    X_natural_transformed
)

natural_contributions = booster.predict(
    natural_dmatrix,
    pred_contribs=True
)

print(
    "Natural contribution shape:",
    natural_contributions.shape
)

if natural_contributions.ndim != 3:
    raise ValueError(
        "Unexpected TreeSHAP shape: "
        f"{natural_contributions.shape}"
    )

natural_feature_contributions = (
    natural_contributions[:, :, :-1]
)

transformed_natural_global = (
    np.abs(natural_feature_contributions)
    .mean(axis=(0, 1))
)

natural_global_importance = (
    aggregate_to_original_features(
        transformed_natural_global,
        original_feature_map
    )
)

print(
    "Original-feature scores:",
    len(natural_global_importance)
)

In [ ]:
corrected_rankings = pd.concat(
    [
        natural_global_importance.rename(
            "natural_global_shap"
        ),
        global_importance.rename(
            "balanced_sample_global_shap"
        ),
        balanced_importance.rename(
            "class_balanced_shap"
        ),
        minority_importance.rename(
            "minority_diagnostic_shap"
        )
    ],
    axis=1
)

score_columns = [
    "natural_global_shap",
    "balanced_sample_global_shap",
    "class_balanced_shap",
    "minority_diagnostic_shap"
]

for score_column in score_columns:
    rank_column = score_column.replace(
        "_shap",
        "_rank"
    )

    corrected_rankings[rank_column] = (
        corrected_rankings[score_column]
        .rank(
            ascending=False,
            method="min"
        )
        .astype(int)
    )

corrected_rankings = corrected_rankings.sort_values(
    "natural_global_shap",
    ascending=False
)

display(corrected_rankings.head(20))

In [ ]:
rank_columns = [
    column
    for column in corrected_rankings.columns
    if column.endswith("_rank")
]

rank_correlations = (
    corrected_rankings[rank_columns]
    .corr(method="spearman")
)

print("Spearman rank correlations:")
display(rank_correlations.round(3))

In [ ]:
corrected_rankings.to_csv(
    "/kaggle/working/"
    "project_xia_corrected_shap_rankings.csv"
)

natural_distribution.to_csv(
    "/kaggle/working/"
    "project_xia_natural_shap_sample_distribution.csv"
)

rank_correlations.to_csv(
    "/kaggle/working/"
    "project_xia_shap_rank_correlations.csv"
)

print("Corrected SHAP evidence saved.")

## Feature-subset comparison and stability analysis

In [ ]:
import time
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_recall_fscore_support
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

# Load rankings if not already present in memory
corrected_rankings = pd.read_csv(
    "/kaggle/working/"
    "project_xia_corrected_shap_rankings.csv"
)

ranking_methods = {
    "Natural_Global": "natural_global_rank",
    "Class_Balanced": "class_balanced_rank",
    "Minority_Diagnostic": "minority_diagnostic_rank"
}

subset_sizes = [5, 10, 15, 20]

fingerprinting_number = list(
    label_encoder.classes_
).index("Fingerprinting")

mitm_number = list(
    label_encoder.classes_
).index("MITM")

In [ ]:
def create_subset_preprocessor(
    selected_features,
    reference_data
):
    numeric_subset = [
        feature
        for feature in selected_features
        if reference_data[feature].dtype != "object"
    ]

    categorical_subset = [
        feature
        for feature in selected_features
        if reference_data[feature].dtype == "object"
    ]

    transformers = []

    if numeric_subset:
        numeric_pipeline = Pipeline(
            steps=[
                (
                    "imputer",
                    SimpleImputer(strategy="median")
                )
            ]
        )

        transformers.append(
            (
                "numeric",
                numeric_pipeline,
                numeric_subset
            )
        )

    if categorical_subset:
        categorical_pipeline = Pipeline(
            steps=[
                (
                    "imputer",
                    SimpleImputer(
                        strategy="most_frequent"
                    )
                ),
                (
                    "encoder",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse=True
                    )
                )
            ]
        )

        transformers.append(
            (
                "categorical",
                categorical_pipeline,
                categorical_subset
            )
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )

In [ ]:
subset_results = []
subset_feature_records = []

for method_name, rank_column in ranking_methods.items():

    ordered_features = (
        corrected_rankings
        .sort_values(rank_column)["feature"]
        .tolist()
    )

    for subset_size in subset_sizes:

        selected_features = ordered_features[
            :subset_size
        ]

        print(
            f"Running {method_name}, "
            f"top {subset_size}..."
        )

        subset_preprocessor = (
            create_subset_preprocessor(
                selected_features,
                X_train
            )
        )

        subset_model = XGBClassifier(
            objective="multi:softprob",
            num_class=len(label_encoder.classes_),
            n_estimators=250,
            max_depth=6,
            learning_rate=0.10,
            subsample=0.80,
            colsample_bytree=0.80,
            min_child_weight=1,
            reg_alpha=0.0,
            reg_lambda=1.0,
            tree_method="hist",
            eval_metric="mlogloss",
            use_label_encoder=False,
            random_state=42,
            n_jobs=-1
        )

        subset_pipeline = Pipeline(
            steps=[
                (
                    "preprocessor",
                    subset_preprocessor
                ),
                (
                    "model",
                    subset_model
                )
            ]
        )

        start_time = time.perf_counter()

        subset_pipeline.fit(
            X_train[selected_features],
            y_train
        )

        training_seconds = (
            time.perf_counter() - start_time
        )

        start_time = time.perf_counter()

        predictions = subset_pipeline.predict(
            X_validation[selected_features]
        )

        inference_seconds = (
            time.perf_counter() - start_time
        )

        precision, recall, class_f1, support = (
            precision_recall_fscore_support(
                y_validation,
                predictions,
                labels=np.arange(
                    len(label_encoder.classes_)
                ),
                zero_division=0
            )
        )

        subset_results.append({
            "method": method_name,
            "subset_size": subset_size,
            "accuracy": accuracy_score(
                y_validation,
                predictions
            ),
            "macro_f1": f1_score(
                y_validation,
                predictions,
                average="macro"
            ),
            "balanced_accuracy":
                balanced_accuracy_score(
                    y_validation,
                    predictions
                ),
            "mcc": matthews_corrcoef(
                y_validation,
                predictions
            ),
            "fingerprinting_recall":
                recall[fingerprinting_number],
            "fingerprinting_f1":
                class_f1[fingerprinting_number],
            "mitm_recall":
                recall[mitm_number],
            "mitm_f1":
                class_f1[mitm_number],
            "training_seconds":
                training_seconds,
            "inference_seconds":
                inference_seconds,
            "milliseconds_per_record":
                inference_seconds /
                len(X_validation) * 1000
        })

        for rank_position, feature in enumerate(
            selected_features,
            start=1
        ):
            subset_feature_records.append({
                "method": method_name,
                "subset_size": subset_size,
                "rank_position": rank_position,
                "feature": feature
            })

results_table = pd.DataFrame(subset_results)

display(
    results_table.sort_values(
        ["subset_size", "macro_f1"],
        ascending=[True, False]
    ).round(4)
)

In [ ]:
subset_features_table = pd.DataFrame(
    subset_feature_records
)

results_table.to_csv(
    "/kaggle/working/"
    "project_xia_shap_subset_pilot_results.csv",
    index=False
)

subset_features_table.to_csv(
    "/kaggle/working/"
    "project_xia_shap_subset_features.csv",
    index=False
)

print("Subset experiment evidence saved.")

In [ ]:
print("Best overall macro-F1 configuration:")

display(
    results_table.nlargest(
        5,
        "macro_f1"
    ).round(4)
)

print("Best Fingerprinting configurations:")

display(
    results_table.sort_values(
        [
            "fingerprinting_recall",
            "fingerprinting_f1",
            "macro_f1"
        ],
        ascending=False
    ).head(5).round(4)
)

print("Best efficiency-performance configurations:")

display(
    results_table.sort_values(
        [
            "subset_size",
            "macro_f1"
        ],
        ascending=[True, False]
    ).head(8).round(4)
)

In [ ]:
import itertools
import time
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

y_development_encoded = label_encoder.transform(
    y_development
)

stability_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fold_ranking_records = []

for fold_number, (train_positions, validation_positions) in enumerate(
    stability_cv.split(
        X_development,
        y_development_encoded
    ),
    start=1
):
    print(f"\nTraining stability fold {fold_number}/5")

    X_fold_train = X_development.iloc[
        train_positions
    ]

    y_fold_train = y_development_encoded[
        train_positions
    ]

    X_fold_validation = X_development.iloc[
        validation_positions
    ]

    y_fold_validation_text = y_development.iloc[
        validation_positions
    ]

    fold_model = XGBClassifier(
        objective="multi:softprob",
        num_class=len(label_encoder.classes_),
        n_estimators=250,
        max_depth=6,
        learning_rate=0.10,
        subsample=0.80,
        colsample_bytree=0.80,
        min_child_weight=1,
        reg_alpha=0.0,
        reg_lambda=1.0,
        tree_method="hist",
        eval_metric="mlogloss",
        use_label_encoder=False,
        random_state=42,
        n_jobs=-1
    )

    fold_pipeline = Pipeline(
        steps=[
            ("preprocessor", clone(preprocessor)),
            ("model", fold_model)
        ]
    )

    fold_pipeline.fit(
        X_fold_train,
        y_fold_train
    )

    fold_preprocessor = (
        fold_pipeline.named_steps["preprocessor"]
    )

    fold_booster = (
        fold_pipeline.named_steps["model"]
        .get_booster()
    )

    fold_encoder = (
        fold_preprocessor
        .named_transformers_["categorical"]
        .named_steps["encoder"]
    )

    # Map expanded columns to original predictors
    fold_original_map = list(numeric_features)

    for feature, categories in zip(
        categorical_features,
        fold_encoder.categories_
    ):
        fold_original_map.extend(
            [feature] * len(categories)
        )

    # Natural-distribution sample for global SHAP
    natural_sample = X_fold_validation.sample(
        n=min(4000, len(X_fold_validation)),
        random_state=100 + fold_number
    )

    natural_transformed = (
        fold_preprocessor.transform(
            natural_sample
        )
    )

    natural_contributions = fold_booster.predict(
        xgb.DMatrix(natural_transformed),
        pred_contribs=True
    )[:, :, :-1]

    natural_scores_transformed = (
        np.abs(natural_contributions)
        .mean(axis=(0, 1))
    )

    # Class-balanced sample for class-specific SHAP
    balanced_indices = []

    for class_number, class_name in enumerate(
        label_encoder.classes_
    ):
        available_indices = (
            y_fold_validation_text[
                y_fold_validation_text == class_name
            ].index
        )

        sample_size = min(
            200,
            len(available_indices)
        )

        sampled = (
            pd.Series(available_indices)
            .sample(
                n=sample_size,
                random_state=(
                    200 +
                    fold_number * 20 +
                    class_number
                )
            )
            .tolist()
        )

        balanced_indices.extend(sampled)

    balanced_sample = X_fold_validation.loc[
        balanced_indices
    ]

    balanced_targets = (
        y_fold_validation_text.loc[
            balanced_indices
        ]
    )

    balanced_targets_encoded = (
        label_encoder.transform(
            balanced_targets
        )
    )

    balanced_transformed = (
        fold_preprocessor.transform(
            balanced_sample
        )
    )

    balanced_contributions = fold_booster.predict(
        xgb.DMatrix(balanced_transformed),
        pred_contribs=True
    )[:, :, :-1]

    class_score_rows = []

    for class_number in range(
        len(label_encoder.classes_)
    ):
        class_mask = (
            balanced_targets_encoded ==
            class_number
        )

        class_scores = np.abs(
            balanced_contributions[
                class_mask,
                class_number,
                :
            ]
        ).mean(axis=0)

        class_score_rows.append(class_scores)

    class_score_matrix = np.vstack(
        class_score_rows
    )

    balanced_scores_transformed = (
        class_score_matrix.mean(axis=0)
    )

    minority_numbers = [
        list(label_encoder.classes_).index(
            "Fingerprinting"
        ),
        list(label_encoder.classes_).index(
            "MITM"
        )
    ]

    minority_scores_transformed = (
        class_score_matrix[
            minority_numbers
        ].mean(axis=0)
    )

    fold_scores = {
        "Natural_Global":
            natural_scores_transformed,
        "Class_Balanced":
            balanced_scores_transformed,
        "Minority_Diagnostic":
            minority_scores_transformed
    }

    for method_name, transformed_scores in (
        fold_scores.items()
    ):
        original_scores = (
            aggregate_to_original_features(
                transformed_scores,
                fold_original_map
            )
        )

        ranks = original_scores.rank(
            ascending=False,
            method="min"
        )

        for feature in original_scores.index:
            fold_ranking_records.append({
                "fold": fold_number,
                "method": method_name,
                "feature": feature,
                "importance": original_scores[
                    feature
                ],
                "rank": int(ranks[feature])
            })

    print(f"Fold {fold_number} completed.")

In [ ]:
fold_rankings = pd.DataFrame(
    fold_ranking_records
)

stability_results = []

for method_name in fold_rankings[
    "method"
].unique():

    method_data = fold_rankings[
        fold_rankings["method"] ==
        method_name
    ]

    rank_matrix = method_data.pivot(
        index="fold",
        columns="feature",
        values="rank"
    )

    rank_correlations = rank_matrix.T.corr(
        method="spearman"
    )

    correlation_values = []

    for first_fold, second_fold in itertools.combinations(
        rank_correlations.index,
        2
    ):
        correlation_values.append(
            rank_correlations.loc[
                first_fold,
                second_fold
            ]
        )

    for subset_size in [5, 10, 15, 20]:

        selected_sets = []

        for fold_number in sorted(
            method_data["fold"].unique()
        ):
            fold_data = method_data[
                method_data["fold"] ==
                fold_number
            ]

            selected = set(
                fold_data.nsmallest(
                    subset_size,
                    "rank"
                )["feature"]
            )

            selected_sets.append(selected)

        jaccard_values = []

        for first_set, second_set in itertools.combinations(
            selected_sets,
            2
        ):
            jaccard_values.append(
                len(first_set & second_set) /
                len(first_set | second_set)
            )

        all_features = sorted(
            method_data["feature"].unique()
        )

        selection_matrix = np.array([
            [
                int(feature in selected_set)
                for feature in all_features
            ]
            for selected_set in selected_sets
        ])

        number_features = len(all_features)
        expected_variance = (
            subset_size / number_features
        ) * (
            1 -
            subset_size / number_features
        )

        observed_variance = (
            selection_matrix.var(
                axis=0,
                ddof=1
            ).mean()
        )

        nogueira_stability = (
            1 -
            observed_variance /
            expected_variance
        )

        stability_results.append({
            "method": method_name,
            "subset_size": subset_size,
            "mean_spearman_rank_correlation":
                np.mean(correlation_values),
            "sd_spearman_rank_correlation":
                np.std(
                    correlation_values,
                    ddof=1
                ),
            "mean_pairwise_jaccard":
                np.mean(jaccard_values),
            "nogueira_stability":
                nogueira_stability
        })

stability_table = pd.DataFrame(
    stability_results
)

display(
    stability_table.sort_values(
        ["subset_size", "nogueira_stability"],
        ascending=[True, False]
    ).round(4)
)

In [ ]:
fold_rankings.to_csv(
    "/kaggle/working/"
    "project_xia_foldwise_shap_rankings.csv",
    index=False
)

stability_table.to_csv(
    "/kaggle/working/"
    "project_xia_shap_stability_results.csv",
    index=False
)

print("Fold-wise SHAP stability evidence saved.")

## CS-SHAP selection

In [ ]:
import pandas as pd
import numpy as np

fold_rankings = pd.read_csv(
    "/kaggle/working/"
    "project_xia_foldwise_shap_rankings.csv"
)

STABILITY_THRESHOLD = 0.80
CANDIDATE_K = 15

def calculate_selection_evidence(
    method_name,
    subset_size
):
    method_data = fold_rankings[
        fold_rankings["method"] ==
        method_name
    ].copy()

    method_data["selected"] = (
        method_data["rank"] <= subset_size
    )

    return (
        method_data.groupby("feature")
        .agg(
            selection_frequency=(
                "selected",
                "mean"
            ),
            mean_rank=("rank", "mean"),
            rank_standard_deviation=(
                "rank",
                "std"
            ),
            mean_importance=(
                "importance",
                "mean"
            )
        )
    )

balanced_evidence = calculate_selection_evidence(
    "Class_Balanced",
    CANDIDATE_K
)

minority_evidence = calculate_selection_evidence(
    "Minority_Diagnostic",
    CANDIDATE_K
)

stable_balanced_core = set(
    balanced_evidence[
        balanced_evidence[
            "selection_frequency"
        ] >= STABILITY_THRESHOLD
    ].index
)

stable_minority_features = set(
    minority_evidence[
        minority_evidence[
            "selection_frequency"
        ] >= STABILITY_THRESHOLD
    ].index
)

minority_additions = (
    stable_minority_features -
    stable_balanced_core
)

cs_shap_features = sorted(
    stable_balanced_core |
    stable_minority_features
)

print(
    "Stable class-balanced core:",
    len(stable_balanced_core)
)

print(
    "Stable minority feature set:",
    len(stable_minority_features)
)

print(
    "Minority additions:",
    len(minority_additions)
)

print(
    "Final provisional CS-SHAP subset:",
    len(cs_shap_features)
)

print("\nCS-SHAP features:")

for number, feature in enumerate(
    cs_shap_features,
    start=1
):
    print(number, feature)

In [ ]:
selection_register = pd.DataFrame(
    index=sorted(
        set(balanced_evidence.index) |
        set(minority_evidence.index)
    )
)

selection_register[
    "class_balanced_frequency"
] = balanced_evidence[
    "selection_frequency"
]

selection_register[
    "class_balanced_mean_rank"
] = balanced_evidence["mean_rank"]

selection_register[
    "minority_frequency"
] = minority_evidence[
    "selection_frequency"
]

selection_register[
    "minority_mean_rank"
] = minority_evidence["mean_rank"]

selection_register["selected"] = (
    selection_register.index.isin(
        cs_shap_features
    )
)

selection_register["selection_role"] = (
    "Not selected"
)

selection_register.loc[
    list(stable_balanced_core),
    "selection_role"
] = "Stable class-balanced core"

selection_register.loc[
    list(minority_additions),
    "selection_role"
] = "Minority-protective addition"

selection_register = (
    selection_register
    .reset_index()
    .rename(columns={"index": "feature"})
    .sort_values(
        [
            "selected",
            "selection_role",
            "class_balanced_mean_rank"
        ],
        ascending=[False, True, True]
    )
)

display(
    selection_register[
        selection_register["selected"]
    ].round(4)
)

In [ ]:
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_recall_fscore_support
)
import time

cs_preprocessor = create_subset_preprocessor(
    cs_shap_features,
    X_train
)

cs_model = XGBClassifier(
    objective="multi:softprob",
    num_class=len(label_encoder.classes_),
    n_estimators=250,
    max_depth=6,
    learning_rate=0.10,
    subsample=0.80,
    colsample_bytree=0.80,
    min_child_weight=1,
    reg_alpha=0.0,
    reg_lambda=1.0,
    tree_method="hist",
    eval_metric="mlogloss",
    use_label_encoder=False,
    random_state=42,
    n_jobs=-1
)

cs_pipeline = Pipeline(
    steps=[
        ("preprocessor", cs_preprocessor),
        ("model", cs_model)
    ]
)

start_time = time.perf_counter()

cs_pipeline.fit(
    X_train[cs_shap_features],
    y_train
)

cs_training_time = (
    time.perf_counter() - start_time
)

start_time = time.perf_counter()

cs_predictions = cs_pipeline.predict(
    X_validation[cs_shap_features]
)

cs_inference_time = (
    time.perf_counter() - start_time
)

precision, recall, class_f1, support = (
    precision_recall_fscore_support(
        y_validation,
        cs_predictions,
        labels=np.arange(
            len(label_encoder.classes_)
        ),
        zero_division=0
    )
)

cs_result = pd.DataFrame([{
    "method": "CS_SHAP_Consensus",
    "subset_size": len(cs_shap_features),
    "accuracy": accuracy_score(
        y_validation,
        cs_predictions
    ),
    "macro_f1": f1_score(
        y_validation,
        cs_predictions,
        average="macro"
    ),
    "balanced_accuracy":
        balanced_accuracy_score(
            y_validation,
            cs_predictions
        ),
    "mcc": matthews_corrcoef(
        y_validation,
        cs_predictions
    ),
    "fingerprinting_recall":
        recall[fingerprinting_number],
    "fingerprinting_f1":
        class_f1[fingerprinting_number],
    "mitm_recall":
        recall[mitm_number],
    "mitm_f1":
        class_f1[mitm_number],
    "training_seconds":
        cs_training_time,
    "inference_seconds":
        cs_inference_time,
    "milliseconds_per_record":
        cs_inference_time /
        len(X_validation) * 1000
}])

display(cs_result.round(4))

In [ ]:
selection_register.to_csv(
    "/kaggle/working/"
    "project_xia_cs_shap_selection_register.csv",
    index=False
)

cs_result.to_csv(
    "/kaggle/working/"
    "project_xia_cs_shap_pilot_result.csv",
    index=False
)

print("CS-SHAP pilot evidence saved.")

In [ ]:
import time
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_recall_fscore_support
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

corrected_rankings = pd.read_csv(
    "/kaggle/working/"
    "project_xia_corrected_shap_rankings.csv"
)

selection_register = pd.read_csv(
    "/kaggle/working/"
    "project_xia_cs_shap_selection_register.csv"
)

natural_top20 = (
    corrected_rankings
    .sort_values("natural_global_rank")
    .head(20)["feature"]
    .tolist()
)

balanced_top15 = (
    corrected_rankings
    .sort_values("class_balanced_rank")
    .head(15)["feature"]
    .tolist()
)

cs_shap_features = (
    selection_register[
        selection_register["selected"] == True
    ]["feature"]
    .tolist()
)

comparison_sets = {
    "Full_42": list(X_development.columns),
    "Natural_Global_20": natural_top20,
    "Class_Balanced_15": balanced_top15,
    "CS_SHAP_19": cs_shap_features
}

for method, features in comparison_sets.items():
    print(method, len(features))

In [ ]:
comparison_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

comparison_records = []

y_dev_encoded = label_encoder.transform(
    y_development
)

for method_name, selected_features in (
    comparison_sets.items()
):
    print(f"\n===== {method_name} =====")

    for fold_number, (
        train_positions,
        validation_positions
    ) in enumerate(
        comparison_cv.split(
            X_development,
            y_dev_encoded
        ),
        start=1
    ):
        print(f"Running fold {fold_number}/5")

        X_fold_train = X_development.iloc[
            train_positions
        ][selected_features]

        X_fold_validation = X_development.iloc[
            validation_positions
        ][selected_features]

        y_fold_train = y_dev_encoded[
            train_positions
        ]

        y_fold_validation = y_dev_encoded[
            validation_positions
        ]

        fold_preprocessor = (
            create_subset_preprocessor(
                selected_features,
                X_fold_train
            )
        )

        fold_model = XGBClassifier(
            objective="multi:softprob",
            num_class=len(
                label_encoder.classes_
            ),
            n_estimators=250,
            max_depth=6,
            learning_rate=0.10,
            subsample=0.80,
            colsample_bytree=0.80,
            min_child_weight=1,
            reg_alpha=0.0,
            reg_lambda=1.0,
            tree_method="hist",
            eval_metric="mlogloss",
            use_label_encoder=False,
            random_state=42,
            n_jobs=-1
        )

        fold_pipeline = Pipeline(
            steps=[
                (
                    "preprocessor",
                    fold_preprocessor
                ),
                ("model", fold_model)
            ]
        )

        start_time = time.perf_counter()

        fold_pipeline.fit(
            X_fold_train,
            y_fold_train
        )

        training_seconds = (
            time.perf_counter() -
            start_time
        )

        start_time = time.perf_counter()

        predictions = fold_pipeline.predict(
            X_fold_validation
        )

        inference_seconds = (
            time.perf_counter() -
            start_time
        )

        precision, recall, class_f1, support = (
            precision_recall_fscore_support(
                y_fold_validation,
                predictions,
                labels=np.arange(
                    len(label_encoder.classes_)
                ),
                zero_division=0
            )
        )

        comparison_records.append({
            "method": method_name,
            "fold": fold_number,
            "feature_count":
                len(selected_features),
            "accuracy": accuracy_score(
                y_fold_validation,
                predictions
            ),
            "macro_f1": f1_score(
                y_fold_validation,
                predictions,
                average="macro"
            ),
            "weighted_f1": f1_score(
                y_fold_validation,
                predictions,
                average="weighted"
            ),
            "balanced_accuracy":
                balanced_accuracy_score(
                    y_fold_validation,
                    predictions
                ),
            "mcc": matthews_corrcoef(
                y_fold_validation,
                predictions
            ),
            "fingerprinting_recall":
                recall[fingerprinting_number],
            "fingerprinting_f1":
                class_f1[
                    fingerprinting_number
                ],
            "mitm_recall":
                recall[mitm_number],
            "mitm_f1":
                class_f1[mitm_number],
            "training_seconds":
                training_seconds,
            "inference_seconds":
                inference_seconds,
            "milliseconds_per_record":
                inference_seconds /
                len(X_fold_validation) *
                1000
        })

In [ ]:
comparison_folds = pd.DataFrame(
    comparison_records
)

metrics = [
    "accuracy",
    "macro_f1",
    "weighted_f1",
    "balanced_accuracy",
    "mcc",
    "fingerprinting_recall",
    "fingerprinting_f1",
    "mitm_recall",
    "mitm_f1",
    "training_seconds",
    "milliseconds_per_record"
]

comparison_summary = (
    comparison_folds
    .groupby(
        ["method", "feature_count"]
    )[metrics]
    .agg(["mean", "std"])
    .reset_index()
)

display(comparison_summary.round(4))

comparison_folds.to_csv(
    "/kaggle/working/"
    "project_xia_fixed_subset_cv_folds.csv",
    index=False
)

comparison_summary.to_csv(
    "/kaggle/working/"
    "project_xia_fixed_subset_cv_summary.csv",
    index=False
)

print("Five-fold comparison saved.")

## Nested cross-validation

Selection is repeated within each outer fold's training data.

In [ ]:
import gc
import itertools
import time
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_recall_fscore_support
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier


def aggregate_scores(scores, original_map):
    table = pd.DataFrame({
        "feature": original_map,
        "importance": scores
    })

    return (
        table.groupby("feature")["importance"]
        .sum()
    )


def create_xia_model(random_state):
    return XGBClassifier(
        objective="multi:softprob",
        num_class=len(label_encoder.classes_),
        n_estimators=250,
        max_depth=6,
        learning_rate=0.10,
        subsample=0.80,
        colsample_bytree=0.80,
        min_child_weight=1,
        reg_alpha=0.0,
        reg_lambda=1.0,
        tree_method="hist",
        eval_metric="mlogloss",
        use_label_encoder=False,
        random_state=random_state,
        n_jobs=-1
    )


def calculate_inner_shap_rankings(
    X_inner_train,
    y_inner_train,
    X_inner_validation,
    y_inner_validation_text,
    random_state
):
    all_features = list(X_inner_train.columns)

    inner_preprocessor = create_subset_preprocessor(
        all_features,
        X_inner_train
    )

    inner_pipeline = Pipeline(
        steps=[
            ("preprocessor", inner_preprocessor),
            (
                "model",
                create_xia_model(random_state)
            )
        ]
    )

    inner_pipeline.fit(
        X_inner_train,
        y_inner_train
    )

    fitted_preprocessor = (
        inner_pipeline.named_steps["preprocessor"]
    )

    booster = (
        inner_pipeline.named_steps["model"]
        .get_booster()
    )

    numeric_local = [
        feature
        for feature in all_features
        if X_inner_train[feature].dtype != "object"
    ]

    categorical_local = [
        feature
        for feature in all_features
        if X_inner_train[feature].dtype == "object"
    ]

    fitted_encoder = (
        fitted_preprocessor
        .named_transformers_["categorical"]
        .named_steps["encoder"]
    )

    original_feature_map = list(numeric_local)

    for feature, categories in zip(
        categorical_local,
        fitted_encoder.categories_
    ):
        original_feature_map.extend(
            [feature] * len(categories)
        )

    # Natural-distribution sample
    natural_sample = X_inner_validation.sample(
        n=min(3000, len(X_inner_validation)),
        random_state=random_state
    )

    natural_transformed = (
        fitted_preprocessor.transform(
            natural_sample
        )
    )

    natural_contributions = booster.predict(
        xgb.DMatrix(natural_transformed),
        pred_contribs=True
    )[:, :, :-1]

    natural_scores = (
        np.abs(natural_contributions)
        .mean(axis=(0, 1))
    )

    # Equal-class sample
    balanced_indices = []

    for class_number, class_name in enumerate(
        label_encoder.classes_
    ):
        available = (
            y_inner_validation_text[
                y_inner_validation_text ==
                class_name
            ].index
        )

        sample_size = min(150, len(available))

        sampled = (
            pd.Series(available)
            .sample(
                n=sample_size,
                random_state=(
                    random_state +
                    class_number +
                    100
                )
            )
            .tolist()
        )

        balanced_indices.extend(sampled)

    balanced_sample = (
        X_inner_validation.loc[
            balanced_indices
        ]
    )

    balanced_target_text = (
        y_inner_validation_text.loc[
            balanced_indices
        ]
    )

    balanced_target = label_encoder.transform(
        balanced_target_text
    )

    balanced_transformed = (
        fitted_preprocessor.transform(
            balanced_sample
        )
    )

    balanced_contributions = booster.predict(
        xgb.DMatrix(balanced_transformed),
        pred_contribs=True
    )[:, :, :-1]

    class_scores = []

    for class_number in range(
        len(label_encoder.classes_)
    ):
        class_mask = (
            balanced_target == class_number
        )

        score = np.abs(
            balanced_contributions[
                class_mask,
                class_number,
                :
            ]
        ).mean(axis=0)

        class_scores.append(score)

    class_scores = np.vstack(class_scores)

    balanced_scores = class_scores.mean(axis=0)

    fingerprinting_number = list(
        label_encoder.classes_
    ).index("Fingerprinting")

    mitm_number = list(
        label_encoder.classes_
    ).index("MITM")

    minority_scores = class_scores[
        [
            fingerprinting_number,
            mitm_number
        ]
    ].mean(axis=0)

    method_scores = {
        "Natural_Global": natural_scores,
        "Class_Balanced": balanced_scores,
        "Minority_Diagnostic": minority_scores
    }

    records = []

    for method_name, transformed_scores in (
        method_scores.items()
    ):
        original_scores = aggregate_scores(
            transformed_scores,
            original_feature_map
        )

        ranks = original_scores.rank(
            ascending=False,
            method="min"
        )

        for feature in original_scores.index:
            records.append({
                "method": method_name,
                "feature": feature,
                "importance":
                    original_scores[feature],
                "rank": int(ranks[feature])
            })

    del inner_pipeline
    del natural_contributions
    del balanced_contributions
    gc.collect()

    return pd.DataFrame(records)


print("Nested-ranking functions created.")

In [ ]:
OUTER_FOLDS = 5
INNER_FOLDS = 5
STABILITY_THRESHOLD = 0.80

y_dev_encoded = label_encoder.transform(
    y_development
)

outer_cv = StratifiedKFold(
    n_splits=OUTER_FOLDS,
    shuffle=True,
    random_state=42
)

nested_results = []
nested_selections = []
inner_ranking_evidence = []

experiment_start = time.perf_counter()

for outer_fold, (
    outer_train_positions,
    outer_validation_positions
) in enumerate(
    outer_cv.split(
        X_development,
        y_dev_encoded
    ),
    start=1
):
    print(
        f"\n========== OUTER FOLD "
        f"{outer_fold}/5 =========="
    )

    X_outer_train = X_development.iloc[
        outer_train_positions
    ]

    X_outer_validation = X_development.iloc[
        outer_validation_positions
    ]

    y_outer_train = y_dev_encoded[
        outer_train_positions
    ]

    y_outer_validation = y_dev_encoded[
        outer_validation_positions
    ]

    y_outer_train_text = y_development.iloc[
        outer_train_positions
    ]

    inner_cv = StratifiedKFold(
        n_splits=INNER_FOLDS,
        shuffle=True,
        random_state=100 + outer_fold
    )

    outer_inner_rankings = []

    for inner_fold, (
        inner_train_positions,
        inner_validation_positions
    ) in enumerate(
        inner_cv.split(
            X_outer_train,
            y_outer_train
        ),
        start=1
    ):
        print(
            f"Outer {outer_fold}: "
            f"inner ranking fold "
            f"{inner_fold}/5"
        )

        X_inner_train = X_outer_train.iloc[
            inner_train_positions
        ]

        X_inner_validation = (
            X_outer_train.iloc[
                inner_validation_positions
            ]
        )

        y_inner_train = y_outer_train[
            inner_train_positions
        ]

        y_inner_validation_text = (
            y_outer_train_text.iloc[
                inner_validation_positions
            ]
        )

        inner_rankings = (
            calculate_inner_shap_rankings(
                X_inner_train,
                y_inner_train,
                X_inner_validation,
                y_inner_validation_text,
                random_state=(
                    1000 +
                    outer_fold * 100 +
                    inner_fold
                )
            )
        )

        inner_rankings["outer_fold"] = (
            outer_fold
        )

        inner_rankings["inner_fold"] = (
            inner_fold
        )

        outer_inner_rankings.append(
            inner_rankings
        )

        inner_ranking_evidence.append(
            inner_rankings
        )

    outer_inner_rankings = pd.concat(
        outer_inner_rankings,
        ignore_index=True
    )

    # Natural-global top 20 using mean inner rank
    natural_summary = (
        outer_inner_rankings[
            outer_inner_rankings["method"] ==
            "Natural_Global"
        ]
        .groupby("feature")
        .agg(
            mean_rank=("rank", "mean"),
            mean_importance=(
                "importance",
                "mean"
            )
        )
        .sort_values(
            ["mean_rank", "mean_importance"],
            ascending=[True, False]
        )
    )

    natural_top20 = (
        natural_summary.head(20)
        .index.tolist()
    )

    # Class-balanced top 15
    balanced_summary = (
        outer_inner_rankings[
            outer_inner_rankings["method"] ==
            "Class_Balanced"
        ]
        .groupby("feature")
        .agg(
            mean_rank=("rank", "mean"),
            mean_importance=(
                "importance",
                "mean"
            )
        )
        .sort_values(
            ["mean_rank", "mean_importance"],
            ascending=[True, False]
        )
    )

    balanced_top15 = (
        balanced_summary.head(15)
        .index.tolist()
    )

    # Stable class-balanced core
    balanced_inner = outer_inner_rankings[
        outer_inner_rankings["method"] ==
        "Class_Balanced"
    ].copy()

    balanced_inner["selected_top15"] = (
        balanced_inner["rank"] <= 15
    )

    balanced_frequency = (
        balanced_inner.groupby("feature")[
            "selected_top15"
        ].mean()
    )

    stable_balanced = set(
        balanced_frequency[
            balanced_frequency >=
            STABILITY_THRESHOLD
        ].index
    )

    # Stable minority-protective features
    minority_inner = outer_inner_rankings[
        outer_inner_rankings["method"] ==
        "Minority_Diagnostic"
    ].copy()

    minority_inner["selected_top15"] = (
        minority_inner["rank"] <= 15
    )

    minority_frequency = (
        minority_inner.groupby("feature")[
            "selected_top15"
        ].mean()
    )

    stable_minority = set(
        minority_frequency[
            minority_frequency >=
            STABILITY_THRESHOLD
        ].index
    )

    cs_features = sorted(
        stable_balanced |
        stable_minority
    )

    comparison_sets = {
        "Full_42":
            list(X_outer_train.columns),
        "Natural_Global_20":
            natural_top20,
        "Class_Balanced_15":
            balanced_top15,
        "CS_SHAP":
            cs_features
    }

    print(
        "Outer-fold feature counts:",
        {
            name: len(features)
            for name, features
            in comparison_sets.items()
        }
    )

    for method_name, selected_features in (
        comparison_sets.items()
    ):
        for feature in selected_features:
            nested_selections.append({
                "outer_fold": outer_fold,
                "method": method_name,
                "feature": feature,
                "feature_count":
                    len(selected_features),
                "stable_balanced": (
                    feature in stable_balanced
                ),
                "stable_minority": (
                    feature in stable_minority
                )
            })

        outer_preprocessor = (
            create_subset_preprocessor(
                selected_features,
                X_outer_train
            )
        )

        outer_pipeline = Pipeline(
            steps=[
                (
                    "preprocessor",
                    outer_preprocessor
                ),
                (
                    "model",
                    create_xia_model(
                        5000 + outer_fold
                    )
                )
            ]
        )

        start_time = time.perf_counter()

        outer_pipeline.fit(
            X_outer_train[selected_features],
            y_outer_train
        )

        training_seconds = (
            time.perf_counter() -
            start_time
        )

        start_time = time.perf_counter()

        predictions = outer_pipeline.predict(
            X_outer_validation[
                selected_features
            ]
        )

        inference_seconds = (
            time.perf_counter() -
            start_time
        )

        precision, recall, class_f1, support = (
            precision_recall_fscore_support(
                y_outer_validation,
                predictions,
                labels=np.arange(
                    len(label_encoder.classes_)
                ),
                zero_division=0
            )
        )

        fingerprinting_number = list(
            label_encoder.classes_
        ).index("Fingerprinting")

        mitm_number = list(
            label_encoder.classes_
        ).index("MITM")

        nested_results.append({
            "outer_fold": outer_fold,
            "method": method_name,
            "feature_count":
                len(selected_features),
            "accuracy": accuracy_score(
                y_outer_validation,
                predictions
            ),
            "macro_f1": f1_score(
                y_outer_validation,
                predictions,
                average="macro"
            ),
            "weighted_f1": f1_score(
                y_outer_validation,
                predictions,
                average="weighted"
            ),
            "balanced_accuracy":
                balanced_accuracy_score(
                    y_outer_validation,
                    predictions
                ),
            "mcc": matthews_corrcoef(
                y_outer_validation,
                predictions
            ),
            "fingerprinting_recall":
                recall[
                    fingerprinting_number
                ],
            "fingerprinting_f1":
                class_f1[
                    fingerprinting_number
                ],
            "mitm_recall":
                recall[mitm_number],
            "mitm_f1":
                class_f1[mitm_number],
            "training_seconds":
                training_seconds,
            "inference_seconds":
                inference_seconds,
            "milliseconds_per_record":
                inference_seconds /
                len(X_outer_validation) *
                1000
        })

        del outer_pipeline
        gc.collect()

    # Save a checkpoint after every completed outer fold
    pd.DataFrame(nested_results).to_csv(
        "/kaggle/working/"
        "project_xia_nested_cv_results_checkpoint.csv",
        index=False
    )

    pd.DataFrame(nested_selections).to_csv(
        "/kaggle/working/"
        "project_xia_nested_selections_checkpoint.csv",
        index=False
    )

    pd.concat(
        inner_ranking_evidence,
        ignore_index=True
    ).to_csv(
        "/kaggle/working/"
        "project_xia_nested_inner_rankings_checkpoint.csv",
        index=False
    )

    elapsed_minutes = (
        time.perf_counter() -
        experiment_start
    ) / 60

    print(
        f"Outer fold {outer_fold} complete. "
        f"Elapsed: {elapsed_minutes:.1f} minutes"
    )

print("\nNested validation completed.")

In [ ]:
nested_results_table = pd.DataFrame(
    nested_results
)

nested_selection_table = pd.DataFrame(
    nested_selections
)

nested_inner_rankings = pd.concat(
    inner_ranking_evidence,
    ignore_index=True
)

summary_metrics = [
    "feature_count",
    "accuracy",
    "macro_f1",
    "weighted_f1",
    "balanced_accuracy",
    "mcc",
    "fingerprinting_recall",
    "fingerprinting_f1",
    "mitm_recall",
    "mitm_f1",
    "training_seconds",
    "milliseconds_per_record"
]

nested_summary = (
    nested_results_table
    .groupby("method")[
        summary_metrics
    ]
    .agg(["mean", "std", "min", "max"])
)

display(nested_summary.round(4))

nested_results_table.to_csv(
    "/kaggle/working/"
    "project_xia_nested_cv_results.csv",
    index=False
)

nested_summary.to_csv(
    "/kaggle/working/"
    "project_xia_nested_cv_summary.csv"
)

nested_selection_table.to_csv(
    "/kaggle/working/"
    "project_xia_nested_feature_selections.csv",
    index=False
)

nested_inner_rankings.to_csv(
    "/kaggle/working/"
    "project_xia_nested_inner_rankings.csv",
    index=False
)

print("Final nested-validation files saved.")

## Frozen feature sets and locked-test evaluation

This is the only stage evaluating the held-out test partition.

In [ ]:
import pandas as pd
import numpy as np
import time
import joblib

corrected_rankings = pd.read_csv(
    "/kaggle/working/"
    "project_xia_corrected_shap_rankings.csv"
)

selection_register = pd.read_csv(
    "/kaggle/working/"
    "project_xia_cs_shap_selection_register.csv"
)

final_feature_sets = {
    "Full_42": list(X_development.columns),

    "Natural_Global_20": (
        corrected_rankings
        .sort_values("natural_global_rank")
        .head(20)["feature"]
        .tolist()
    ),

    "Class_Balanced_15": (
        corrected_rankings
        .sort_values("class_balanced_rank")
        .head(15)["feature"]
        .tolist()
    ),

    "CS_SHAP_19": (
        selection_register[
            selection_register["selected"] == True
        ]["feature"]
        .tolist()
    )
}

for method, features in final_feature_sets.items():
    print(method, len(features))
    print(features)
    print()

In [ ]:
frozen_feature_records = []

for method, features in final_feature_sets.items():
    for position, feature in enumerate(
        features,
        start=1
    ):
        frozen_feature_records.append({
            "method": method,
            "feature_count": len(features),
            "position": position,
            "feature": feature
        })

frozen_feature_table = pd.DataFrame(
    frozen_feature_records
)

frozen_feature_table.to_csv(
    "/kaggle/working/"
    "project_xia_final_frozen_feature_sets.csv",
    index=False
)

print("Feature sets frozen.")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_recall_fscore_support,
    roc_auc_score
)
from sklearn.preprocessing import label_binarize
from sklearn.pipeline import Pipeline

y_development_encoded = label_encoder.transform(
    y_development
)

y_test_encoded = label_encoder.transform(
    y_test
)

y_test_binary = label_binarize(
    y_test_encoded,
    classes=np.arange(
        len(label_encoder.classes_)
    )
)

locked_test_results = []
per_class_records = []
prediction_records = []
confusion_records = []

final_cs_pipeline = None

for method_name, selected_features in (
    final_feature_sets.items()
):
    print(f"\nEvaluating {method_name}...")

    final_preprocessor = create_subset_preprocessor(
        selected_features,
        X_development
    )

    final_pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                final_preprocessor
            ),
            (
                "model",
                create_xia_model(42)
            )
        ]
    )

    start_time = time.perf_counter()

    final_pipeline.fit(
        X_development[selected_features],
        y_development_encoded
    )

    training_seconds = (
        time.perf_counter() - start_time
    )

    start_time = time.perf_counter()

    test_predictions = final_pipeline.predict(
        X_test[selected_features]
    )

    test_probabilities = (
        final_pipeline.predict_proba(
            X_test[selected_features]
        )
    )

    inference_seconds = (
        time.perf_counter() - start_time
    )

    precision, recall, class_f1, support = (
        precision_recall_fscore_support(
            y_test_encoded,
            test_predictions,
            labels=np.arange(
                len(label_encoder.classes_)
            ),
            zero_division=0
        )
    )

    macro_pr_auc = average_precision_score(
        y_test_binary,
        test_probabilities,
        average="macro"
    )

    weighted_pr_auc = average_precision_score(
        y_test_binary,
        test_probabilities,
        average="weighted"
    )

    macro_roc_auc = roc_auc_score(
        y_test_binary,
        test_probabilities,
        average="macro",
        multi_class="ovr"
    )

    locked_test_results.append({
        "method": method_name,
        "feature_count": len(selected_features),
        "accuracy": accuracy_score(
            y_test_encoded,
            test_predictions
        ),
        "macro_f1": f1_score(
            y_test_encoded,
            test_predictions,
            average="macro"
        ),
        "weighted_f1": f1_score(
            y_test_encoded,
            test_predictions,
            average="weighted"
        ),
        "balanced_accuracy":
            balanced_accuracy_score(
                y_test_encoded,
                test_predictions
            ),
        "mcc": matthews_corrcoef(
            y_test_encoded,
            test_predictions
        ),
        "macro_pr_auc": macro_pr_auc,
        "weighted_pr_auc": weighted_pr_auc,
        "macro_roc_auc_ovr": macro_roc_auc,
        "multiclass_log_loss": log_loss(
            y_test_encoded,
            test_probabilities,
            labels=np.arange(
                len(label_encoder.classes_)
            )
        ),
        "training_seconds":
            training_seconds,
        "inference_seconds":
            inference_seconds,
        "milliseconds_per_record":
            inference_seconds /
            len(X_test) * 1000
    })

    for class_number, class_name in enumerate(
        label_encoder.classes_
    ):
        binary_actual = (
            y_test_encoded == class_number
        ).astype(int)

        class_ap = average_precision_score(
            binary_actual,
            test_probabilities[:, class_number]
        )

        per_class_records.append({
            "method": method_name,
            "class": class_name,
            "precision":
                precision[class_number],
            "recall":
                recall[class_number],
            "f1":
                class_f1[class_number],
            "support":
                support[class_number],
            "average_precision":
                class_ap
        })

    actual_names = label_encoder.inverse_transform(
        y_test_encoded
    )

    predicted_names = label_encoder.inverse_transform(
        test_predictions
    )

    for original_index, actual, predicted in zip(
        X_test.index,
        actual_names,
        predicted_names
    ):
        prediction_records.append({
            "method": method_name,
            "original_index": original_index,
            "actual_class": actual,
            "predicted_class": predicted,
            "correct": actual == predicted
        })

    from sklearn.metrics import confusion_matrix

    cm = confusion_matrix(
        y_test_encoded,
        test_predictions,
        labels=np.arange(
            len(label_encoder.classes_)
        )
    )

    for actual_number, actual_class in enumerate(
        label_encoder.classes_
    ):
        for predicted_number, predicted_class in enumerate(
            label_encoder.classes_
        ):
            confusion_records.append({
                "method": method_name,
                "actual_class": actual_class,
                "predicted_class":
                    predicted_class,
                "count": int(
                    cm[
                        actual_number,
                        predicted_number
                    ]
                )
            })

    if method_name == "CS_SHAP_19":
        final_cs_pipeline = final_pipeline

    print(
        method_name,
        "completed in",
        round(training_seconds, 2),
        "seconds"
    )

In [ ]:
locked_test_table = pd.DataFrame(
    locked_test_results
)

per_class_table = pd.DataFrame(
    per_class_records
)

locked_predictions_table = pd.DataFrame(
    prediction_records
)

confusion_table = pd.DataFrame(
    confusion_records
)

display(
    locked_test_table.sort_values(
        "macro_f1",
        ascending=False
    ).round(5)
)

print("\nFingerprinting and MITM results:")

display(
    per_class_table[
        per_class_table["class"].isin(
            ["Fingerprinting", "MITM"]
        )
    ].sort_values(
        ["class", "method"]
    ).round(5)
)

locked_test_table.to_csv(
    "/kaggle/working/"
    "project_xia_locked_test_results.csv",
    index=False
)

per_class_table.to_csv(
    "/kaggle/working/"
    "project_xia_locked_test_per_class.csv",
    index=False
)

locked_predictions_table.to_csv(
    "/kaggle/working/"
    "project_xia_locked_test_predictions.csv",
    index=False
)

confusion_table.to_csv(
    "/kaggle/working/"
    "project_xia_locked_test_confusion_matrix.csv",
    index=False
)

joblib.dump(
    final_cs_pipeline,
    "/kaggle/working/"
    "project_xia_final_cs_shap_pipeline.joblib"
)

print("Final locked-test evidence saved.")